[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lucascamillomd/pyaging/blob/main/tutorials/tutorial_bloodchemistry.ipynb) [![Open In nbviewer](https://img.shields.io/badge/View%20in-nbviewer-orange)](https://nbviewer.jupyter.org/github/lucascamillomd/pyaging/blob/main/tutorials/tutorial_bloodchemistry.ipynb)

# Blood chemistry

`pyaging` ships four clocks that run on a routine blood panel plus a couple of clinical measurements: `phenoage`, `kdmage`, `homeostaticdysregulation` and `linage2`.

We just need two packages for this tutorial.

In [1]:
import pandas as pd
import pyaging as pya

### Pre-release note (remove after the v0.5.0 upload)

Hugging Face still serves the v0.4.x clocks and example data, so the cells below read the local copies instead.

In [2]:
import shutil
from pathlib import Path

from pyaging.predict import _pred_utils

build = Path('../clocks')
if build.is_dir():
    _pred_utils.download_clock_weights = lambda name, *args, **kwargs: str(build / 'weights' / f'{name}.pt')
    Path('pyaging_data').mkdir(exist_ok=True)
    shutil.copy(build / 'blood_chemistry_example.pkl', 'pyaging_data/blood_chemistry_example.pkl')
    print('Pre-release: using the clocks and example data built in this checkout, not Hugging Face.')

Pre-release: using the clocks and example data built in this checkout, not Hugging Face.


## Download and load example data

30 real NHANES IV subjects, from the public `NHANES4` table of the R package [BioAge](https://github.com/dayoonkwon/BioAge).

In [3]:
pya.data.download_example_data('blood_chemistry_example')

⏺ example data already at pyaging_data/blood_chemistry_example.pkl

'pyaging_data/blood_chemistry_example.pkl'

In [4]:
df = pd.read_pickle('pyaging_data/blood_chemistry_example.pkl')

In [5]:
df.head()

,albumin,creatinine,glucose,c_reactive_protein,lymphocyte_percent,mean_cell_volume,red_cell_distribution_width,alkaline_phosphatase,white_blood_cell_count,total_cholesterol,blood_urea_nitrogen,hemoglobin_a1c,systolic_blood_pressure,forced_expiratory_volume,age,female
NHANES_2007_41475,36.0,45.968884,5.8830,1.98,20.0,92.1,13.4,113.0,8.5,4.62894,4.641,5.7,123.333333,2.025,62.0,1.0
NHANES_2007_41479,42.0,63.649224,5.4945,0.02,46.4,87.6,12.1,78.0,5.1,4.86168,4.284,5.7,108.666667,2.957,52.0,0.0
NHANES_2007_41482,45.0,61.881190,7.9365,0.40,22.3,89.4,12.4,85.0,9.4,4.08588,3.927,7.0,116.000000,2.920,64.0,0.0
NHANES_2007_41483,35.0,120.226312,5.9940,1.49,24.0,85.3,14.7,55.0,9.8,3.67212,6.069,6.5,109.333333,2.355,66.0,0.0
NHANES_2007_41486,43.0,54.809054,6.2160,0.39,36.3,90.7,11.7,101.0,5.6,5.01684,3.213,6.1,122.666667,2.364,61.0,1.0


`pya.utils.get_feature_ranges` shows the unit and plausible range for each feature.

In [6]:
pya.utils.get_feature_ranges('PhenoAge')

,feature,unit,low,high
0,albumin,g/L,10.00,70.0
1,creatinine,umol/L,10.00,3000.0
2,glucose,mmol/L,1.00,60.0
3,c_reactive_protein,mg/dL,0.01,50.0
4,lymphocyte_percent,%,0.00,100.0
5,mean_cell_volume,fL,40.00,150.0
6,red_cell_distribution_width,%,8.00,40.0
7,alkaline_phosphatase,U/L,5.00,5000.0
8,white_blood_cell_count,10^3 cells/uL,0.05,500.0
9,age,years,0.00,122.5


## Convert data to AnnData object

AnnData objects are highly flexible and are thus our preferred method of organizing data for age prediction.

In [7]:
adata = pya.preprocess.df_to_adata(df)

Note that the original DataFrame is stored in `X_original` under layers. This is what the `adata` object looks like:

In [8]:
adata

AnnData object with n_obs × n_vars = 30 × 16
    var: 'percent_na'
    layers: 'X_original', None (.X)

## Predict age

LinAge2 is included deliberately even though this panel does not cover it, so you can see what an incomplete input looks like.

In [9]:
pya.pred.predict_age(adata, ['PhenoAge', 'KDMAge', 'HomeostaticDysregulation', 'LinAge2'])

In [10]:
adata.obs.head()

,phenoage,kdmage,homeostaticdysregulation,linage2
NHANES_2007_41475,72.931468,54.722088,4.302934,60.837872
NHANES_2007_41479,45.493712,41.957925,3.207144,46.463145
NHANES_2007_41482,70.980976,48.563729,4.277576,56.877960
NHANES_2007_41483,86.424060,72.901056,4.843122,70.061409
NHANES_2007_41486,58.511456,47.659222,3.486106,54.568415


`phenoage` and `kdmage` are in years; `homeostaticdysregulation` is a log dysregulation score, not an age. LinAge2 warned that most of its features were missing — treat that estimate as unusable.

## Running LinAge2 on a panel it actually covers

LinAge2 needs 57 measurements **plus 26 questionnaire items**. Here are the two subjects the paper publishes, which carry the full panel.

In [11]:
from pathlib import Path

# The paper's two published example subjects, lifted from its supplementary archive and renamed
# to pyaging's feature names by clocks/notebooks/linage2.ipynb.
subjects = 'linage2_published_subjects.csv'
if not Path(subjects).is_file():
    subjects = f'https://raw.githubusercontent.com/lucascamillomd/pyaging/main/tutorials/{subjects}'

wide = pd.read_csv(subjects, index_col=0)
wide.iloc[:, :8]

,told_high_blood_pressure,told_diabetes,general_health_condition,health_compared_to_one_year_ago,healthcare_visits_past_year,hospital_overnight_past_year,told_weak_or_failing_kidneys,told_asthma
sample,,,,,,,,
NHANES_8881,1.0,1.0,3.0,1.0,3.0,2.0,2.0,2.0
NHANES_9106,2.0,2.0,2.0,3.0,2.0,2.0,2.0,2.0


In [12]:
linage2_adata = pya.preprocess.df_to_adata(wide, verbose=False)
pya.pred.predict_age(linage2_adata, 'LinAge2')
linage2_adata.obs

,linage2
sample,
NHANES_8881,88.694487
NHANES_9106,64.357899


No missing-feature warning, and both match the 88.69 and 64.36 years the paper reports.

## Units matter, and pyaging tells you when they don't line up

Albumin is reported in g/L by some labs and g/dL by others. `pyaging` expects g/L.

In [13]:
in_g_per_dl = df.copy()
in_g_per_dl['albumin'] = in_g_per_dl['albumin'] / 10  # g/L -> g/dL, the wrong unit for pyaging

wrong_units = pya.preprocess.df_to_adata(in_g_per_dl, verbose=False)
pya.pred.predict_age(wrong_units, 'PhenoAge')

The check is warn-only — it never blocks a prediction — but it names the feature, the expected range, and what it saw.

In [14]:
comparison = pd.DataFrame({'albumin in g/L (correct)': adata.obs['phenoage'], 'albumin in g/dL (wrong)': wrong_units.obs['phenoage']})
comparison['error in years'] = comparison.iloc[:, 1] - comparison.iloc[:, 0]
comparison.head()

,albumin in g/L (correct),albumin in g/dL (wrong),error in years
NHANES_2007_41475,72.931468,85.005332,12.073864
NHANES_2007_41479,45.493712,59.579887,14.086175
NHANES_2007_41482,70.980976,86.073306,15.092330
NHANES_2007_41483,86.424060,98.162539,11.738479
NHANES_2007_41486,58.511456,72.933016,14.421560


Around fourteen years of phenotypic age from one misread unit, with no error and no NaN to tip you off. Feeding the column back in g/L makes the warning and the inflation go away.

In [15]:
corrected = in_g_per_dl.copy()
corrected['albumin'] = corrected['albumin'] * 10  # back to g/L

fixed = pya.preprocess.df_to_adata(corrected, verbose=False)
pya.pred.predict_age(fixed, 'PhenoAge')
fixed.obs.head()

,phenoage
NHANES_2007_41475,72.931468
NHANES_2007_41479,45.493712
NHANES_2007_41482,70.980976
NHANES_2007_41483,86.424060
NHANES_2007_41486,58.511456


## Turning the output down

Having so much information printed can be overwhelming, particularly when running several clocks at once. In such cases, just set verbose to False.

In [16]:
pya.data.download_example_data('blood_chemistry_example', verbose=False)
df = pd.read_pickle('pyaging_data/blood_chemistry_example.pkl')
adata = pya.preprocess.df_to_adata(df, verbose=False)
pya.pred.predict_age(adata, ['PhenoAge', 'KDMAge', 'HomeostaticDysregulation'], verbose=False)

In [17]:
adata.obs.head()

,phenoage,kdmage,homeostaticdysregulation
NHANES_2007_41475,72.931468,54.722088,4.302934
NHANES_2007_41479,45.493712,41.957925,3.207144
NHANES_2007_41482,70.980976,48.563729,4.277576
NHANES_2007_41483,86.424060,72.901056,4.843122
NHANES_2007_41486,58.511456,47.659222,3.486106


`verbose=False` silences the range warnings too, so keep it on while you are still getting to know a dataset.

In [18]:
adata

AnnData object with n_obs × n_vars = 30 × 16
    obs: 'phenoage', 'kdmage', 'homeostaticdysregulation'
    var: 'percent_na'
    uns: 'phenoage_percent_na', 'phenoage_missing_features', 'phenoage_supplied_features_mask', 'phenoage_metadata', 'kdmage_percent_na', 'kdmage_missing_features', 'kdmage_supplied_features_mask', 'kdmage_metadata', 'homeostaticdysregulation_percent_na', 'homeostaticdysregulation_missing_features', 'homeostaticdysregulation_supplied_features_mask', 'homeostaticdysregulation_metadata'
    layers: 'X_original', None (.X)

## Get citation

The doi, citation, and some metadata are automatically added to the AnnData object under `adata.uns[CLOCKNAME_metadata]`.

In [19]:
adata.uns['kdmage_metadata']

{'clock_name': 'kdmage',
 'data_type': 'clinical biomarkers',
 'species': 'Homo sapiens',
 'year': 2021,
 'approved_by_author': '⌛',
 'citation': 'Kwon, Dayoon, and Daniel W. Belsky. "A toolkit for quantification of biological age from blood chemistry and organ function test data: BioAge." GeroScience 43.6 (2021): 2795-2808.',
 'doi': 'https://doi.org/10.1007/s11357-021-00480-5',
 'notes': "Klemera-Doubal biological age, trained sex-specifically on NHANES III adults aged 30-75 who were not pregnant, using the BioAge package defaults. Biomarker parameters were fit on SI-unit variants so they are natively in pyaging's unit convention, and C-reactive protein is supplied raw in mg/dL and log1p-transformed inside the clock. Sex is coded female = 1 and male = 0; a dataset with no female column scores every sample with the male parameters.",
 'research_only': None,
 'version': None,
 'tissue': ['blood'],
 'predicts': ['biological age'],
 'training_target': ['chronological age'],
 'unit': ['ye